# Data Analysis for BSSE Project

This notebook contains Ryan's analysis of Ibrahim's data. 

## Reading in Data

The goal of this section is to read the data into data structures that are conducive to analysis. At the same time, we want to build in sanity checks to help us ensure that calculations run successfully and that calculations are what we think they are. The end goal is to have the following data structures:

- Map from geometries to "short names"
- Table of energies with one row per "short name"

To define the short names:

- `{system}_{parameters}` where `{system}` is one or more underscore-separated monomers, e.g., `Ne_Ne_{parameters}` is a neon dimer.
-  Monomers in `()` are ghosted, i.e., `Ne_(Ne)_{parameters}` computes the energy of the first Ne atom using the dimer basis set.

Misc. Notes.

- Data for the project lives in `bsse_db/data/monomer_name`, where "monomer_name" is the molecular formula of the monomers in the cluster.
- We are going to assume that for a monomer containing $n$ atoms, the first $n$ atoms in a file belong to monomer 1, the next $n$ belong to monomer 2, the next $n$ belong to monomer 3, etc.
- Ghost atoms in NWChem are specified by prepending `bq` to the atomic symbol.
- Initial data allowed the molecular system to be reoriented. This leads to small geometric differences between the supersystem and subsystem geometries. 


In [1]:
import tarfile
import itertools
import os
import math
import pandas as pd
from nwchem_helpers.parse_nwchem_output import parse_nwchem_output
from nwchem_helpers.compare_results import similar_nwchem_runs

def parse_tarred_file(tarball, file_path):
    f = tarball.extractfile(file_path)
    content = f.read().decode("utf-8").split('\n')
    return parse_nwchem_output(iter(content))

def save_state(key, new_results, collected_results):
    if key in collected_results:
        assert similar_nwchem_runs(collected_results[key], new_results)
    else:
        collected_results[key] = new_results

def make_key_from_file(monomers, file_name):
    A, B = monomers
    if file_name == 'output_E_AB_AB.txt':
        return '{}_{}'.format(A, B)
    elif file_name == 'output_E_AB_A.txt':
        return '{}_({})'.format(A, B)
    elif file_name == 'output_E_AB_B.txt':
        return '({})_{}'.format(A, B)
    elif file_name == 'output_E_A_A.txt':
        return '{}'.format(A)
    else:
        raise Exception("Unrecognized filename")

### Ne Clusters

- `Ne_dimers.tar` contain CCSD(T)/aug-cc-pvdz calculations.
- `Ne_Ne_distance_x_y_z` directory contains a dimer where one of the monomers has been translated by $\vec{r} = (x,y,z)^T$.
- Translating like this duplicates effort because of the system's symmetry (i.e., only the distance between the Neons matters)
- No 'output_E_B_B.txt' because monomers are the same.

In [3]:
nsteps    = 7    # The total number of displacements along each axis
step_size = 0.25 # How much we displace for each step.

ne_dimer_distances = set()
ne_dimer_results = {}

def compute_ne_distance(geom):
    carts = [[float(geom[i][j]) for j in range(1, 4)] for i in range(2)]
    return math.dist(carts[0], carts[1])

with tarfile.open('data/Ne/Ne_dimers.tar', 'r') as tarball:
    all_names = tarball.getnames() # Gets all the directories and files inside the tarball
    
    for dx, dy, dz in itertools.product(range(1, nsteps), range(1, nsteps), range(1, nsteps)):
        x, y, z = (dx * step_size, dy * step_size, dz * step_size)
        directory_name = os.path.join('Ne_dimers', 'Ne_Ne_distance_{}_{}_{}'.format(x, y, z))

        monomers = ('Ne', 'Ne')
        if directory_name in all_names: # Checks translation is in tarball
            
            # Extract the results from the dimer file
            dimer_file_name = 'output_E_AB_AB.txt'
            file_name = os.path.join(directory_name, dimer_file_name)
            results = parse_tarred_file(tarball, file_name)
            
            # Compute and record the separation distance
            geom = results['Input Geometry (angstroms)']
            r = compute_ne_distance(geom)
            ne_dimer_distances.add(r)

            # Save the state
            key = make_key_from_file(monomers, dimer_file_name) + '_{}'.format(r)
            save_state(key, results, ne_dimer_results)

            for monomer_file in ['output_E_AB_A.txt', 'output_E_AB_B.txt']:
                file_name = os.path.join(directory_name, monomer_file)
                results = parse_tarred_file(tarball, file_name)
    
                # Sanity check it's the same distance
                geom = results['Input Geometry (angstroms)']
                r_new = compute_ne_distance(geom)
                assert math.isclose(r, r_new, abs_tol=1E-7) # NWChem only prints about 8 decimal places

                # Save the state (use dimer distance for consistency)
                key = make_key_from_file(monomers, monomer_file) + '_{}'.format(r)
                save_state(key, results, ne_dimer_results)

            for monomer_file in ['output_E_A_A.txt']:
                file_name = os.path.join(directory_name, monomer_file)
                results = parse_tarred_file(tarball, file_name)
                key = make_key_from_file(monomers, monomer_file)
                save_state(key, results, ne_dimer_results)
                
pd.DataFrame(ne_dimer_results).transpose()

,Input Geometry (angstroms),AO Basis Set,Total SCF Energy (a.u.),Total MP2 Energy (a.u.),Total CCSD Energy (a.u.),Total CCSD(T) Energy (a.u.)
Ne_Ne_1.5411035,"[(Ne, 0.00000000, 0.00000000, -0.77055175), (N...",aug-cc-pvdz,-256.874113911105,-257.292842423520881,-257.299248528910653,-257.305108143114182
Ne_(Ne)_1.5411035,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvdz,-128.496953904925,-128.706906455561153,-128.710024487065823,-128.712905546045221
(Ne)_Ne_1.5411035,"[(bqNe, 0.00000000, 0.00000000, -1.54110350), ...",aug-cc-pvdz,-128.496953904925,-128.706906455561125,-128.710024474655171,-128.712905531504191
Ne,"[(Ne, 0.00000000, 0.00000000, 0.00000000)]",aug-cc-pvdz,-128.496349730514,-128.705409599288487,-128.708487837634010,-128.711294152566950
Ne_Ne_1.60078106,"[(Ne, 0.00000000, 0.00000000, -0.80039053), (N...",aug-cc-pvdz,-256.902970297102,-257.321600541022690,-257.328049289253840,-257.333887071280117
...,...,...,...,...,...,...
Ne_(Ne)_2.46221446,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvdz,-128.496439816931,-128.705570759155108,-128.708655551024947,-128.711479578227227
(Ne)_Ne_2.46221446,"[(bqNe, 0.00000000, 0.00000000, -2.46221445), ...",aug-cc-pvdz,-128.496439816931,-128.705570759155023,-128.708655541533858,-128.711479570102568
Ne_Ne_2.59807622,"[(Ne, 0.00000000, 0.00000000, -1.29903811), (N...",aug-cc-pvdz,-256.991935285700,-257.410324413582885,-257.416530286745228,-257.422199193337235
Ne_(Ne)_2.59807622,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvdz,-128.496428160688,-128.705549214647675,-128.708635670441282,-128.711457349988592
